# Register every fish of one experiment

Runs the plane → own-anatomy registration of `register_plane_to_anatomy.ipynb` for all fish of one
experiment, skipping fish that are already done. For each fish it writes, into the fish folder
(`02_reg/07_2pf-a/`), the transforms, their metadata and a table of ROI positions, plus the canonical
anatomy NRRD. Small QC figures and a summary table go to `Paper/2p_derived/registration/<experiment>/`.

Use `register_plane_to_anatomy.ipynb` to look at one fish in detail.

## 1. Imports

In [ ]:
%load_ext autoreload
%autoreload 2

import gc
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Walk up to the repo root (marked by utils.py) so the project modules import from anywhere.
repo_root = Path.cwd()
while not (repo_root / "utils.py").exists() and repo_root != repo_root.parent:
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from registration.batch_registration import (
    collect_fish_summary,
    find_experiment_fish_folders,
    fish_status,
    list_experiment_fish,
    qc_figure_paths,
    register_fish,
)
from registration.qc_plotting import (
    draw_depth_profile,
    draw_matched_anatomy,
    draw_mean_image,
    draw_registration_overlay,
    draw_roi_overlay,
    save_figure,
)

## 2. Settings
Choose the experiment here; paths are set per user.

In [ ]:
EXPERIMENT = "Exp_9_RF_memory_inhibition"

LAB_ROOT = Path(r"C:\Users\suribear\OneDrive - Université de Lausanne\Lab")
DATA_ROOT = LAB_ROOT / "Data" / "2p"  # one folder per fish, named "<fish_id>..."
OVERVIEW_CSV = LAB_ROOT / "Results" / "Paper" / "2p_derived" / "global" / "overview.csv"
QC_DIR = LAB_ROOT / "Results" / "Paper" / "2p_derived" / "registration" / EXPERIMENT

QC_DPI = 60  # low resolution keeps each QC image small
GRID_COLUMNS = 3  # plane panels per row in the QC figures

## 3. Which fish are done
Lists the experiment's fish (without `exclude`) and whether each still needs registering.

In [ ]:
experiment_fish = list_experiment_fish(OVERVIEW_CSV, EXPERIMENT)
fish_folders = find_experiment_fish_folders(DATA_ROOT, experiment_fish)
status_table = pd.DataFrame([
    {**fish, "folder": fish_folders[fish["fish_id"]].name if fish_folders[fish["fish_id"]] else None,
     "status": fish_status(fish_folders[fish["fish_id"]], fish["fish_id"], QC_DIR)}
    for fish in experiment_fish
])
status_table

## 4. QC figure layouts
One figure per QC view, with one panel per plane. Edit sizes or colours here.

In [ ]:
def plane_grid(n_planes, panel_size):
    """Create a grid with one panel per plane and hide the unused panels.

    Args:
        n_planes (int): Number of planes.
        panel_size (tuple): (width, height) of one panel, in inches.

    Returns:
        tuple: (figure, list of the first n_planes axes).
    """
    n_rows = -(-n_planes // GRID_COLUMNS)  # ceil division
    figure, axes = plt.subplots(n_rows, GRID_COLUMNS, figsize=(panel_size[0] * GRID_COLUMNS, panel_size[1] * n_rows), squeeze=False, layout="compressed")
    for axis in axes.flat[n_planes:]:
        axis.axis("off")
    plane_axes = list(axes.flat[:n_planes])
    return figure, plane_axes


def make_qc_figures(plane_results, fish_id):
    """Build the four QC figures of one fish.

    Args:
        plane_results (list): From `register_fish`.
        fish_id (str): Fish identifier, used in the titles.

    Returns:
        dict: figure name (matching `qc_figure_paths`) -> matplotlib Figure.
    """
    figures = {}
    for name, draw, panel_size in (
        ("registration_overlays", draw_registration_overlay, (5, 5)),
        ("depth_profiles", draw_depth_profile, (4, 2.8)),
        ("roi_overlays", draw_roi_overlay, (5, 5)),
    ):
        figure, plane_axes = plane_grid(len(plane_results), panel_size)
        for axis, plane_result in zip(plane_axes, plane_results):
            draw(axis, plane_result)
        figure.suptitle(f"{fish_id} — {name.replace('_', ' ')}")
        figures[name] = figure

    # Suite2p mean image next to its matched anatomy, one row per plane.
    figure, axes = plt.subplots(len(plane_results), 2, figsize=(8, 4 * len(plane_results)), squeeze=False, layout="compressed")
    for (mean_axis, anatomy_axis), plane_result in zip(axes, plane_results):
        draw_mean_image(mean_axis, plane_result)
        draw_matched_anatomy(anatomy_axis, plane_result)
    figure.suptitle(f"{fish_id} — mean image vs matched anatomy")
    figures["mean_vs_anatomy"] = figure
    return figures

## 5. Register the remaining fish
Only `todo` fish are registered. A fish that can't be processed (no or several anatomy files, an anatomy still being averaged, missing metadata) is skipped with a warning; rerun this cell later to pick it up. A `qc_missing` fish is already registered and is not redone automatically: delete its `_roi_positions.csv` to force a rerun. Takes a few minutes per fish.

In [ ]:
skipped_fish = {}
for fish in experiment_fish:
    fish_id, fish_folder = fish["fish_id"], fish_folders[fish["fish_id"]]
    current_status = fish_status(fish_folder, fish_id, QC_DIR)
    if current_status == "done":
        continue
    if current_status == "no_folder":
        skipped_fish[fish_id] = f"no single data folder starting with {fish_id} in {DATA_ROOT}"
        continue
    if current_status == "qc_missing":
        skipped_fish[fish_id] = "registered, but QC figures missing; delete its _roi_positions.csv to rerun"
        continue
    print(f"=== {fish_id} ({fish_folder.name})")
    try:
        plane_results = register_fish(fish_folder, fish_id)
        figure_paths = qc_figure_paths(QC_DIR, fish_id)
        for name, figure in make_qc_figures(plane_results, fish_id).items():
            save_figure(figure, figure_paths[name], QC_DPI)
    except (ValueError, RuntimeError, OSError, KeyError, IndexError) as error:
        print(f"WARNING {fish_id}: skipped -- {error}")
        skipped_fish[fish_id] = str(error)
        plt.close("all")  # drop any half-drawn QC figures
        continue
    finally:
        plane_results = None  # images of every plane: free them before the next fish
        gc.collect()
print(f"Skipped {len(skipped_fish)} fish.")

## 6. Summary
One row per fish and plane, read from the saved registration metadata (so it includes fish done in earlier runs). Saved next to the QC figures.

In [ ]:
summary_rows = []
for fish_id, fish_folder in fish_folders.items():
    if fish_folder is not None:
        summary_rows.extend(collect_fish_summary(fish_folder, fish_id))
registration_summary = pd.DataFrame(summary_rows)
if not registration_summary.empty:
    QC_DIR.mkdir(parents=True, exist_ok=True)
    registration_summary.to_csv(QC_DIR / f"{EXPERIMENT}_registration_summary.csv", index=False)
registration_summary.round(3)

## 7. Skipped fish
Why each fish wasn't registered in this run.

In [ ]:
pd.Series(skipped_fish, name="reason", dtype=str)